# 5. Habituation sessions with left video — the eid list to run LightningPose on

Same job as `2_training_session_filter.ipynb` / `3_biased_session_filter.ipynb`: take the
LDA cohort, pick one timepoint, get the eids. The timepoint here is habituationChoiceWorld
and the only requirement is that a raw leftCamera mp4 is registered. LightningPose has not
been run on a single habituation session in this cohort, so its absence is not a filter —
it is the reason the list exists.

Unlike notebooks 2 and 3, **this one writes no CSV**. `habituation_sessions.csv` (from
`4_habituation_sessions_video_qc.py`) already has one row per habituation session with
`mouse_name`, `eid` and `date` in the first three columns, so it *is* the eid list and the
two lists below are one boolean filter each. Writing them out again would only create copies
that go stale the next time the query is re-run. The last cell exports one if a job runner
insists on a pre-filtered file.

Two things the list does not promise, both of which matter downstream:

* **No right camera, anywhere.** Not one habituation session in the cohort has a
  rightCamera or bodyCamera mp4, so the two-camera lick detection in `merge_licks` has no
  counterpart. Whatever is built on these is a one-camera analysis.
* **Under half are time-aligned.** Only some sessions carry ALF `_ibl_leftCamera.times.npy`,
  i.e. frame times already aligned to the task. Without it LP output can be produced but not
  binned into trial epochs until times are extracted from the raw `.timestamps.ssv` / GPIO
  (127 of 152 sessions have that file, so it is recoverable — just a step).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

SESSIONS_CSV = Path.cwd() / 'habituation_sessions.csv'
if not SESSIONS_CSV.exists():
    raise FileNotFoundError(
        f'{SESSIONS_CSV.name} not found. Run 4_habituation_sessions_video_qc.py first '
        '(it queries Alyx and caches the result, so that only has to happen once).')

sessions = pd.read_csv(SESSIONS_CSV)
print(f'{len(sessions)} habituation sessions, {sessions["mouse_name"].nunique()} mice')
print('the LDA cohort is 56 mice; the rest have no habituation session registered on Alyx '
      '-- see habituation_per_mouse.csv, which keeps a row for them')

152 habituation sessions, 53 mice
the LDA cohort is 56 mice; the rest have no habituation session registered on Alyx -- see habituation_per_mouse.csv, which keeps a row for them


## The list: habituation protocol + left video available

`has_raw_leftCam` is the one filter — `_iblrig_leftCamera.raw.mp4` registered on the
session, which is what LP takes as input. `has_leftCam_times` splits it into the part that
can be binned into trials today and the part that needs timestamp extraction first.

In [2]:
LP_COLS = ['mouse_name', 'eid', 'date', 'alyx_lab', 'task_protocol',
           'has_leftCam_times', 'has_rig_timestamps',
           'video_qc_run', 'videoLeft_qc', 'no_quality_FAIL', 'no_timing_FAIL']

lp_list = (sessions.loc[sessions['has_raw_leftCam'], LP_COLS]
           .sort_values(['alyx_lab', 'mouse_name', 'date']).reset_index(drop=True))
lp_timed = lp_list.loc[lp_list['has_leftCam_times']].reset_index(drop=True)

print(f'LP job list          : {len(lp_list):3d} sessions, '
      f'{lp_list["mouse_name"].nunique():2d} mice')
print(f'  already time-aligned: {len(lp_timed):3d} sessions, '
      f'{lp_timed["mouse_name"].nunique():2d} mice')
print(f'  needs times extracted: {len(lp_list) - len(lp_timed):3d} sessions')
print(f'dropped {int((~sessions["has_raw_leftCam"]).sum())} habituation sessions with no '
      'mp4 registered')
lp_list.head(10)

LP job list          : 144 sessions, 51 mice
  already time-aligned:  71 sessions, 28 mice
  needs times extracted:  73 sessions
dropped 8 habituation sessions with no mp4 registered


,mouse_name,eid,date,alyx_lab,task_protocol,has_leftCam_times,has_rig_timestamps,video_qc_run,videoLeft_qc,no_quality_FAIL,no_timing_FAIL
0,NYU-37,ef0e43b5-543a-4f90-b4c4-935484e678a1,2020-11-09,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,False,True,True,FAIL,False,False
1,NYU-37,0de17d0f-e721-42ba-93e8-40059277689e,2020-11-10,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,False,True,False,ABSENT,True,True
2,NYU-37,d701b350-b0a6-4a82-9507-343f6610a836,2020-11-11,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,False,True,True,FAIL,False,False
3,NYU-39,f0545d94-53c0-49e4-a4cc-b69ba5518fcf,2021-02-08,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,False,True,False,ABSENT,True,True
4,NYU-39,092c37ec-196a-47fa-bdfb-1b1389ff4805,2021-02-09,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,False,True,True,FAIL,False,False
5,NYU-39,71ab046e-6927-47ef-9837-17006272bccd,2021-02-10,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,False,True,False,ABSENT,True,True
6,NYU-40,cfacda20-0bf7-496f-8d5c-5f2602e319e2,2021-02-08,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,False,True,False,ABSENT,True,True
7,NYU-40,22279b98-1b40-40b3-b634-160cdabcb26c,2021-02-09,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,False,True,False,ABSENT,True,True
8,NYU-40,ca87ffd4-4dda-4e24-80a0-4b6c18b1d5a6,2021-02-10,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,False,True,False,ABSENT,True,True
9,NYU-45,193545cc-6052-4c58-971d-6314a52e98bb,2021-04-21,angelakilab,_iblrig_tasks_habituationChoiceWorld6.4.2,True,True,True,FAIL,False,False


In [3]:
# The eids themselves, if that is all the LP job wants.
eids = lp_list['eid'].tolist()
print(len(eids), 'eids')
eids[:5]

144 eids


['ef0e43b5-543a-4f90-b4c4-935484e678a1',
 '0de17d0f-e721-42ba-93e8-40059277689e',
 'd701b350-b0a6-4a82-9507-343f6610a836',
 'f0545d94-53c0-49e4-a4cc-b69ba5518fcf',
 '092c37ec-196a-47fa-bdfb-1b1389ff4805']

## Per-mouse coverage

How many sessions each mouse contributes, and how many of those are already time-aligned.
The second column is what decides the sample size of any within-habituation analysis: doing
the timestamp-extraction step is the difference between the two counts printed below.

In [4]:
per_mouse = (lp_list.groupby('mouse_name')
             .agg(lab=('alyx_lab', 'first'), n_video=('eid', 'size'),
                  n_timed=('has_leftCam_times', 'sum'))
             .sort_values(['lab', 'mouse_name']))

print(f'mice with >= 3 sessions with video  : {int((per_mouse["n_video"] >= 3).sum())}')
print(f'mice with >= 3 already time-aligned : {int((per_mouse["n_timed"] >= 3).sum())}')
per_mouse

mice with >= 3 sessions with video  : 39
mice with >= 3 already time-aligned : 17


,lab,n_video,n_timed
mouse_name,,,
NYU-37,angelakilab,3,0
NYU-39,angelakilab,3,0
NYU-40,angelakilab,3,0
NYU-45,angelakilab,3,3
NYU-46,angelakilab,3,3
NYU-47,angelakilab,3,3
NYU-48,angelakilab,1,1
NYU-65,angelakilab,3,0
CSHL045,churchlandlab,4,4


## What the video QC says about the list

Not a filter — reported so a bad LP run can be attributed rather than guessed at.
`ABSENT` means the check was never run on that session, which is a different claim from a
failure; `NO_OUTCOME` means a measurement was stored but no verdict was ever written.

`focus`, `brightness` and `position` are the ones to watch: they bear on whether pose
estimation will track anything, and they are also where the FAILs are. They are deliberately
not filtered on — how well LP tracks habituation video is the empirical question, and these
thresholds were set for a different one.

In [5]:
CHECKS = ['_videoLeft_focus', '_videoLeft_position', '_videoLeft_brightness',
          '_videoLeft_resolution', '_videoLeft_file_headers', '_videoLeft_camera_times',
          '_videoLeft_timestamps', '_videoLeft_pin_state', '_videoLeft_dropped_frames',
          '_videoLeft_framerate']

in_list = sessions.loc[sessions['has_raw_leftCam']]
qc = pd.DataFrame({c: in_list[c].value_counts() for c in CHECKS}).T
qc = qc.reindex(columns=[c for c in ['PASS', 'WARNING', 'FAIL', 'CRITICAL', 'NOT_SET',
                                     'NO_OUTCOME', 'ABSENT'] if c in qc.columns])
print(f'video QC over the {len(in_list)} sessions in the list:')
qc.fillna(0).astype(int)

video QC over the 144 sessions in the list:


,PASS,WARNING,FAIL,NOT_SET,NO_OUTCOME,ABSENT
_videoLeft_focus,60,0,30,0,0,54
_videoLeft_position,66,6,18,0,0,54
_videoLeft_brightness,36,30,24,0,0,54
_videoLeft_resolution,90,0,0,0,0,54
_videoLeft_file_headers,90,0,0,0,0,54
_videoLeft_camera_times,73,3,0,0,14,54
_videoLeft_timestamps,81,0,9,0,0,54
_videoLeft_pin_state,19,1,0,56,14,54
_videoLeft_dropped_frames,19,0,1,56,14,54
_videoLeft_framerate,54,0,22,0,14,54


## Write the list out for the LP job

`habituation_eids.csv` — the handoff file, named the way `first_training_eids.csv` and
`biased_before_ephys_1_eids.csv` are named. **All 144 sessions with an mp4**, not just the
71 time-aligned ones: LP only needs the video, and camera times are a later requirement for
binning into trial epochs, not an input to pose estimation. Dropping the other 73 here would
throw away pose data that is recoverable as soon as times are extracted.

This is a copy of a filter on `habituation_sessions.csv`, so it goes stale the moment the
Alyx query is re-run. Re-run this cell after `4_habituation_sessions_video_qc.py`, and treat
`habituation_sessions.csv` as the source of truth if the two ever disagree.

In [ ]:
lp_list.to_csv('habituation_eids.csv', index=False)
print(f'wrote habituation_eids.csv: {len(lp_list)} sessions, '
      f'{lp_list["mouse_name"].nunique()} mice  <- send this one for LP processing')

# Bare one-eid-per-line version, if the job runner wants that instead of a table.
# Path('habituation_eids.txt').write_text('\n'.join(lp_list['eid']) + '\n')

wrote habituation_eids.csv: 144 sessions, 51 mice  <- send this one for LP processing
